In [1]:
import os
import gc
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import anndata as ad
import scanpy as sc

%matplotlib inline

/home/deepak/pixi_envs/scenic/.pixi/envs/default/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/deepak/pixi_envs/scenic/.pixi/envs/default/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/deepak/pixi_envs/scenic/.pixi/envs/default/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/home/deepak/pixi_envs/scenic/.pixi/envs/default/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/home/deepak/pixi_envs/scenic/.pixi/envs/default/l

In [2]:
## Known Genes In Deleted Regions ##

GENES_22q11 = ["TBX1",  "COMT",  "DGCR8",    "CRKL",  "PRODH",  "HIRA",  "LZTR1",    "RTN4R",   "SNAP29",    "ZDHHC8"]

GENES_22q11_EXTENDED = ["RANBP1", "MED15",  "PI4KA",  "SLC25A1", "SEPT5", "GP1BB", "ZNF74",   
                     "CLDN5",    "ARVCF", "FAM230"]

GENES_3q29 = ["DLG1", "PAK2", "FBXO45", "RNF168", "TFRC", "BDH1", "PCYT1A", "ATP13A3"]

GENES_3q29_EXTENDED = ["WDR53", "CEP19",  "TM4SF1",  "SENP5", "MFI2", "UBXN7"]

GENES_15q13 = ["CHRNA7", "OTUD7A", "KLF13", "TRPM1", "FAN1", "MTMR10", "ARHGAP11B"]

GENES_15q13_EXTNEDED = ["POLR2M", "GOLGA8A", "GOLGA8B", "GOLGA8C", "GOLGA8D", "GOLGA8E", "GOLGA8F", "GOLGA8G", "GOLGA8H", "GOLGA8I"]

In [3]:
## Read in DEG Lists ##

CSV_PATH = '/mnt/sdb/scz_meta_analysis_processed/dge_signatures/dreamlet_dges/disease_analyses/'

dge_22q11 = pd.read_csv(Path(CSV_PATH) / 'dreamlet_22q11_dge_scz_results.csv')
dge_NRXN1 = pd.read_csv(Path(CSV_PATH) / 'dreamlet_NRNXN1_dge_scz_results.csv')
dge_3q29 = pd.read_csv(Path(CSV_PATH) / 'dreamlet_3q29_dge_scz_results.csv')
dge_15q13 = pd.read_csv(Path(CSV_PATH) / 'dreamlet_15q13_dge_scz_results.csv')
dge_Idiopathic = pd.read_csv(Path(CSV_PATH) / 'dreamlet_Idiopathic_dge_scz_results.csv')

In [4]:
UNIQUE_22q11 = np.unique(dge_22q11[(dge_22q11['adj.P.Val'] < 0.05)]['ID'])

UNIQUE_NRXN1 = np.unique(dge_NRXN1[(dge_NRXN1['adj.P.Val'] < 0.05)]['ID'])

UNIQUE_3q29 = np.unique(dge_3q29[(dge_3q29['adj.P.Val'] < 0.05)]['ID'])

UNIQUE_15q13 = np.unique(dge_15q13[(dge_15q13['adj.P.Val'] < 0.05)]['ID'])

UNIQUE_Idiopathic = np.unique(dge_Idiopathic[(dge_Idiopathic['adj.P.Val'] < 0.05)]['ID'])

In [5]:
## Pairwise Combinations

gene_sets = {"22q11": set(UNIQUE_22q11), "NRXN1": set(UNIQUE_NRXN1), 
             "3q29": set(UNIQUE_3q29),    "15q13": set(UNIQUE_15q13),
             "Idiopathic": set(UNIQUE_Idiopathic),}

for (name1, set1), (name2, set2) in combinations(gene_sets.items(), 2):
    overlap = set1 & set2
    print(f"{name1} ∩ {name2}: {len(overlap)} genes")
    if len(overlap) > 0:
        print(overlap)
        print()

22q11 ∩ NRXN1: 0 genes
22q11 ∩ 3q29: 0 genes
22q11 ∩ 15q13: 0 genes
22q11 ∩ Idiopathic: 2 genes
{'PHC1', 'SSC4D'}

NRXN1 ∩ 3q29: 0 genes
NRXN1 ∩ 15q13: 0 genes
NRXN1 ∩ Idiopathic: 0 genes
3q29 ∩ 15q13: 0 genes
3q29 ∩ Idiopathic: 1 genes
{'FBXO45'}

15q13 ∩ Idiopathic: 0 genes


In [6]:
## What percentage of the Known Genes in the Chr Block are DE ##

## make a heatmap for this ##

## 22q11 ##

print('22q11')
print(np.unique(dge_22q11[dge_22q11['adj.P.Val'] < 0.05]['ID'][dge_22q11[dge_22q11['adj.P.Val'] < 0.05]['ID'].isin(GENES_22q11)]))
print('22q11 extended')
print(np.unique(dge_22q11[dge_22q11['adj.P.Val'] < 0.05]['ID'][dge_22q11[dge_22q11['adj.P.Val'] < 0.05]['ID'].isin(GENES_22q11_EXTENDED)]))
print('\n')

## 3q29 ##

print('3q29')
print(np.unique(dge_3q29[dge_3q29['adj.P.Val'] < 0.05]['ID'][dge_3q29[dge_3q29['adj.P.Val'] < 0.05]['ID'].isin(GENES_3q29)]))
print('3q29 extended')
print(np.unique(dge_3q29[dge_3q29['adj.P.Val'] < 0.05]['ID'][dge_3q29[dge_3q29['adj.P.Val'] < 0.05]['ID'].isin(GENES_3q29_EXTENDED)]))
print('\n')

## NRXN1 ##

print('NRXN1')
print(np.unique(dge_NRXN1[dge_NRXN1['adj.P.Val'] < 0.05]['ID'][dge_NRXN1[dge_NRXN1['adj.P.Val'] < 0.05]['ID'].isin(['NRXN1'])]))
print('\n')

## 15q13 ##

print('15q13')
print(np.unique(dge_15q13[dge_15q13['adj.P.Val'] < 0.05]['ID'][dge_15q13[dge_15q13['adj.P.Val'] < 0.05]['ID'].isin([GENES_15q13])]))
print('15q13 extended')
print(np.unique(dge_15q13[dge_15q13['adj.P.Val'] < 0.05]['ID'][dge_15q13[dge_15q13['adj.P.Val'] < 0.05]['ID'].isin([GENES_15q13_EXTNEDED])]))
print('\n')

## Idiopathic ##

print('Idiopathic')
print(np.unique(dge_Idiopathic[dge_Idiopathic['adj.P.Val'] < 0.05]['ID'][dge_Idiopathic[dge_Idiopathic['adj.P.Val'] < 0.05]['ID'].isin(GENES_22q11)]))
print(np.unique(dge_Idiopathic[dge_Idiopathic['adj.P.Val'] < 0.05]['ID'][dge_Idiopathic[dge_Idiopathic['adj.P.Val'] < 0.05]['ID'].isin(GENES_22q11_EXTENDED)]))
print(np.unique(dge_Idiopathic[dge_Idiopathic['adj.P.Val'] < 0.05]['ID'][dge_Idiopathic[dge_Idiopathic['adj.P.Val'] < 0.05]['ID'].isin(GENES_3q29)]))
print(np.unique(dge_Idiopathic[dge_Idiopathic['adj.P.Val'] < 0.05]['ID'][dge_Idiopathic[dge_Idiopathic['adj.P.Val'] < 0.05]['ID'].isin(GENES_3q29_EXTENDED)]))
print(np.unique(dge_Idiopathic[dge_Idiopathic['adj.P.Val'] < 0.05]['ID'][dge_Idiopathic[dge_Idiopathic['adj.P.Val'] < 0.05]['ID'].isin(GENES_15q13)]))
print(np.unique(dge_Idiopathic[dge_Idiopathic['adj.P.Val'] < 0.05]['ID'][dge_Idiopathic[dge_Idiopathic['adj.P.Val'] < 0.05]['ID'].isin(GENES_15q13_EXTNEDED)]))
print(np.unique(dge_Idiopathic[dge_Idiopathic['adj.P.Val'] < 0.05]['ID'][dge_Idiopathic[dge_Idiopathic['adj.P.Val'] < 0.05]['ID'].isin(['NRXN1'])]))

## Idiopathic SCZ Shares 2 genes in Common with 3q29 deletion and Nothing Else ##

22q11
['COMT' 'CRKL' 'HIRA' 'LZTR1' 'RTN4R' 'SNAP29' 'ZDHHC8']
22q11 extended
['ARVCF' 'MED15' 'PI4KA' 'RANBP1' 'SLC25A1']


3q29
['DLG1' 'FBXO45' 'PCYT1A' 'RNF168']
3q29 extended
[]


NRXN1
[]


15q13
[]
15q13 extended
[]


Idiopathic
[]
[]
['FBXO45' 'TFRC']
[]
[]
[]
[]


In [7]:
dge_22q11[(dge_22q11['adj.P.Val'] < 0.05) & (np.abs(dge_22q11['logFC']) > 1)]

,assay,ID,logFC,AveExpr,t,P.Value,adj.P.Val,B,z.std
1,Mixed Neurons FGF12+GRIN2B+CAMK2B+,DGCR6L,-1.019591,5.543906,-7.256137,3.989719e-10,0.000023,3.565761,-6.254429
2,Mixed Neurons FGF12+GRIN2B+CAMK2B+,MED15,-1.002709,4.571850,-7.285524,6.377276e-10,0.000024,8.850007,-6.180812
12,Radial Glia SOX2+VIM+FABP7+,ARVCF,-1.067759,3.905471,-6.292424,2.921863e-08,0.000264,4.588398,-5.546055
24,Mesenchymal-like cells VIM+VCAN+SPARC+,C6orf47,1.048778,3.883652,5.583191,3.014912e-07,0.001401,6.321828,5.122512
30,Radial Glia SOX2+VIM+FABP7+,HIRA,-1.091463,3.156952,-5.788587,6.400092e-07,0.002430,0.877696,-4.978736
33,Mesenchymal-like cells VIM+VCAN+SPARC+,IFI27,1.996876,3.392458,5.380695,9.719776e-07,0.003364,2.424913,4.897229
34,Radial Glia SOX2+VIM+FABP7+,THY1,1.402161,3.037023,5.512177,1.038335e-06,0.003491,2.626528,4.884231
42,Radial Glia SOX2+PAX6+FABP7+,SLC25A1,-1.004481,5.840882,-5.008478,3.128498e-06,0.008562,0.488708,-4.662198
43,Radial Glia SOX2+PAX6+FABP7+,COMT,-2.127382,4.210205,-4.919259,3.844694e-06,0.010088,-0.865093,-4.619606
52,Mesenchymal-like cells VIM+VCAN+SPARC+,PIK3CA,-1.510031,4.096901,-4.998349,1.072027e-05,0.023803,4.060725,-4.402111


In [22]:
dge_22q11[dge_22q11['adj.P.Val'] < 0.05].groupby('assay').count()[['ID']]

,ID
assay,
Dorsal Forebrain Neuron FOXG1+EMX1+NEUROG1+,11
Hindbrain Neurons NR2F2+PBX3+LHX1+,14
IPC EOMES+NEUROG2+PAX6+,2
Mesenchymal-like cells VIM+VCAN+SPARC+,5
Mixed Neurons FGF12+GRIN2B+CAMK2B+,9
Proliferative Radial Glia SOX2+HES6+TOP2A+,10
Radial Glia SOX2+PAX6+FABP7+,9
Radial Glia SOX2+VIM+FABP7+,10


In [10]:
dge_NRXN1[dge_NRXN1['adj.P.Val'] < 0.05].groupby('assay').count()

,ID,logFC,AveExpr,t,P.Value,adj.P.Val,B,z.std
assay,,,,,,,,


In [19]:
dge_3q29[dge_3q29['adj.P.Val'] < 0.05].groupby('assay').count()[['ID']]

,ID
assay,
Astrocyte GFAP+AQP4+HOPX+,1
Dorsal Forebrain Neuron FOXG1+EMX1+NEUROG1+,4
GABAergic GAD1+GAD2+CALB2+,4
Hindbrain Neurons NR2F2+PBX3+LHX1+,7
Mesenchymal-like cells VIM+VCAN+SPARC+,1
Mixed Neurons FGF12+GRIN2B+CAMK2B+,6
Oligodendrocyte OLIG1+OLIG2+MBP+,1
Proliferative Radial Glia SOX2+HES6+TOP2A+,6
Radial Glia SOX2+PAX6+FABP7+,6


In [20]:
dge_15q13[dge_15q13['adj.P.Val'] < 0.05].groupby('assay').count()[['ID']]

,ID
assay,
Dorsal Forebrain Neuron FOXG1+EMX1+NEUROG1+,3
GABAergic GAD1+GAD2+CALB2+,1
Hindbrain Neurons NR2F2+PBX3+LHX1+,3
Mixed Neurons FGF12+GRIN2B+CAMK2B+,1


In [21]:
dge_Idiopathic[dge_Idiopathic['adj.P.Val'] < 0.05].groupby('assay').count()[['ID']]

,ID
assay,
Dorsal Forebrain Neuron FOXG1+EMX1+NEUROG1+,150
GABAergic GAD1+GAD2+CALB2+,52
Hindbrain Neurons NR2F2+PBX3+LHX1+,96
IPC EOMES+NEUROG2+PAX6+,17
Mesenchymal-like cells VIM+VCAN+SPARC+,154
Mixed Neurons FGF12+GRIN2B+CAMK2B+,22
OPC OLIG1+OLIG2+PDGFRA+,35
Proliferative Radial Glia SOX2+HES6+TOP2A+,447
Radial Glia SOX2+PAX6+FABP7+,132


In [16]:
print(dge_22q11[(dge_22q11['adj.P.Val'] < 0.05) & (dge_22q11['logFC'] > 1)]['ID'].shape)

print(dge_NRXN1[(dge_NRXN1['adj.P.Val'] < 0.05) & (dge_NRXN1['logFC'] > 1)]['ID'].shape)

print(dge_3q29[(dge_3q29['adj.P.Val'] < 0.05) & (dge_3q29['logFC'] > 1)]['ID'].shape)

print(dge_15q13[(dge_15q13['adj.P.Val'] < 0.05) & (dge_15q13['logFC'] > 1)]['ID'].shape)

print(dge_Idiopathic[(dge_Idiopathic['adj.P.Val'] < 0.05) & (dge_Idiopathic['logFC'] > 1)]['ID'].shape)

(4,)
(0,)
(8,)
(7,)
(741,)


In [17]:
print(np.unique(dge_22q11[(dge_22q11['adj.P.Val'] < 0.05) & (dge_22q11['logFC'] > 1)]['ID']).tolist())

print(np.unique(dge_NRXN1[(dge_NRXN1['adj.P.Val'] < 0.05) & (dge_NRXN1['logFC'] > 1)]['ID'].tolist()))

print(np.unique(dge_3q29[(dge_3q29['adj.P.Val'] < 0.05) & (dge_3q29['logFC'] > 1)]['ID'].tolist()))

print(np.unique(dge_15q13[(dge_15q13['adj.P.Val'] < 0.05) & (dge_15q13['logFC'] > 1)]['ID'].tolist()))

print(np.unique(dge_Idiopathic[(dge_Idiopathic['adj.P.Val'] < 0.05) & (dge_Idiopathic['logFC'] > 1)]['ID']).tolist())

['C6orf47', 'IFI27', 'SSC4D', 'THY1']
[]
['CHCHD2' 'CTSF' 'ELAPOR2' 'LPAR2' 'NLRP2' 'SEM1']
['BEX5' 'POTEI' 'TCEAL5']
['ABCA7', 'ABCB10', 'ABCC10', 'ABHD14B', 'ABHD16A', 'ACADS', 'ADA', 'ADAM19', 'ADAM8', 'ADAMTS4', 'ADAMTSL4', 'ADCY7', 'ADCYAP1R1', 'ADIPOR1', 'AGAP4', 'AGAP6', 'AHCY', 'ALAD', 'ALDH16A1', 'ALDH2', 'AMACR', 'ANGEL1', 'ANKDD1A', 'ANKFY1', 'ANKH', 'ANKLE1', 'ANKMY1', 'ANKRD9', 'ANLN', 'ANXA2R', 'APOBEC3C', 'APOC1', 'APOE', 'ARHGAP8', 'ARHGEF11', 'ARID3A', 'ARL4D', 'ARL8B', 'ARMC7', 'ARMT1', 'ARPC1B', 'ASB1', 'ASH2L', 'ASIC4', 'ASNS', 'ATAD3C', 'ATF5', 'ATG14', 'ATL2', 'ATL3', 'ATOX1', 'ATP1B2', 'ATP6V1E1', 'ATP6V1FNB', 'ATRIP', 'AXIN1', 'AZU1', 'B3GNT2', 'B4GALT4', 'BAMBI', 'BCL11B', 'BCL2L1', 'BCL2L11', 'BET1L', 'BLOC1S3', 'BLVRB', 'BMP2', 'BOP1', 'BRI3BP', 'BRPF3', 'BTD', 'C15orf39', 'C18orf25', 'C1orf74', 'C2orf88', 'C6orf163', 'C8orf58', 'CABLES2', 'CAD', 'CAPN1', 'CAVIN1', 'CCDC106', 'CCDC24', 'CCDC28B', 'CCNA2', 'CCNB1', 'CCND3', 'CCNDBP1', 'CDC6', 'CDC7', 'CDCA5', 